In [ ]:
"""
Objetivo: Extrair séries temporais macroeconômicas da API do Federal Reserve (FRED),
harmonizar a frequência dos dados para trimestres completos e exportar a base limpa.
"""

import pandas as pd
import pandas_datareader.data as web
import datetime
import warnings
import json
import os
import sys

warnings.filterwarnings('ignore')

def carregar_configuracao(caminho_arquivo: str = "config_pt.json") -> dict:
    """Lê os parâmetros e variáveis a partir de um arquivo JSON externo."""
    if not os.path.exists(caminho_arquivo):
        print(f"[ERRO] Arquivo de configuração '{caminho_arquivo}' não encontrado.")
        sys.exit(1)
        
    try:
        with open(caminho_arquivo, 'r', encoding='utf-8') as arquivo:
            config = json.load(arquivo)
            if "series_fred" not in config or "parametros_gerais" not in config:
                print("[ERRO] O arquivo config.json está com chaves obrigatórias faltando.")
                sys.exit(1)
            return config
    except Exception as e:
        print(f"[ERRO] Falha ao ler o arquivo JSON de configuração: {e}")
        sys.exit(1)

def extrair_dados_fred(dicionario_series: dict, dt_inicio: str, dt_fim: str) -> pd.DataFrame:
    """Consome a API do FRED para baixar as séries temporais e as consolida em um único DataFrame."""
    print(f"[*] Iniciando extração de {len(dicionario_series)} séries do FRED...")
    lista_series = []
    
    for ticker, nome_coluna in dicionario_series.items():
        try:
            print(f" - Baixando ticker: {ticker} ({nome_coluna})...")
            serie = web.DataReader(ticker, 'fred', dt_inicio, dt_fim)
            s_limpa = pd.to_numeric(serie[ticker], errors='coerce')
            s_limpa.name = nome_coluna
            lista_series.append(s_limpa)
        except Exception as e:
            print(f"[AVISO] Falha ao baixar a série {ticker}: {e}")
            
    if not lista_series:
        print("[ERRO] Nenhuma série foi baixada com sucesso do FRED.")
        sys.exit(1)
            
    return pd.concat(lista_series, axis=1)

def harmonizar_frequencia_trimestral(df: pd.DataFrame) -> pd.DataFrame:
    """Converte os dados diários/mensais para médias trimestrais e descarta o trimestre incompleto atual."""
    print("[*] Harmonizando base para frequência trimestral (Média)...")
    df_trimestral = df.resample('QS').mean()
    
    trimestre_atual = pd.Timestamp(datetime.datetime.today()).to_period('Q').start_time
    if trimestre_atual in df_trimestral.index:
        df_trimestral = df_trimestral.drop(trimestre_atual)
        print(f"[*] Trimestre atual ({trimestre_atual.date()}) removido para evitar dados parciais.")
    
    return df_trimestral.dropna(how='all')

if __name__ == "__main__":
    # 1. Carrega parâmetros
    config = carregar_configuracao("config_pt.json")
    inicio = config["parametros_gerais"]["start_date"]
    fim = datetime.datetime.today().strftime('%Y-%m-%d')
    arquivo_saida = config["parametros_gerais"]["output_file"]
    
    # 2. Executa pipeline de dados
    df_bruto = extrair_dados_fred(config["series_fred"], inicio, fim)
    df_final = harmonizar_frequencia_trimestral(df_bruto)
    
    # 3. Exporta resultados
    df_final.to_csv(arquivo_saida, index=True)
    
    print(f"\n[SUCESSO] Base final exportada como '{arquivo_saida}'.")
    print(f"[*] Dimensões do Painel Macro: {df_final.shape[0]} trimestres e {df_final.shape[1]} variáveis.")